# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [0]:
%pip install -q --upgrade "mlflow[databricks]"

In [0]:
import json
from typing import Any

import mlflow

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
EXPERIMENT_NAME = "/Users/sara.martin.delbarrio@gmail.com/semana01-agentops"
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})

2026/10/02 21:13:56 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados.
**RESPUESTA** Estaba ya completo, las palabras relacionadas con secretos, contraseñas, tokens y privacidad llevan a la ruta security; las relacionadas con producción, API y servicios llevan a production; y el resto de preguntas se dirigen a mlflow.


In [0]:
@mlflow.trace
def route_question(question: str) -> str:
    text = question.lower()
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

2026/10/02 21:14:01 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


answer,question
"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",¿Qué objeto agrupa varios runs?
"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",¿Puedo pegar mi token en un parámetro de MLflow?
"Ruta: production. En producción importan contratos, versionado y monitorización.",¿Qué cambia cuando llevo un modelo a producción?


[Trace(trace_id=tr-d15ab74f2bba6ad740c8380a812f9a38), Trace(trace_id=tr-ed5eb9b4cfcef455c553613abab3173f), Trace(trace_id=tr-d1d38c92aca22b037313290d2746a9bb)]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.


Trace ID: tr-d15ab74f2bba6ad740c8380a812f9a38

Ruta: mlflow.

Pregunta:¿Qué objeto agrupa varios runs?

Contexto recuperado Un experimento agrupa runs con parámetros, métricas y artefactos.

Si la ruta fuese incorrecta, cambiaría las palabras clave definidas en route_question para que la pregunta se dirigiera a la ruta adecuada.

## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.


In [0]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    # TODO 2: añade dos casos más y comprueba que siguen pasando.
    {"question": "¿Qué es importante al poner un modelo en producción?","must_include": "versionado"}
    ,
    {"question": "¿Es seguro compartir mi token?", "must_include": "secretos"}

]

evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

question,must_include,answer,passed
¿Qué objeto agrupa varios runs?,experimento,"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",true
¿Puedo guardar una contraseña como tag?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo cuidar en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Qué es importante al poner un modelo en producción?,versionado,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Es seguro compartir mi token?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true


[Trace(trace_id=tr-2a1c8a00f266de8b93338b7d4c2d4156), Trace(trace_id=tr-4316ea217828d7dd71452264ec57bbca), Trace(trace_id=tr-c46d3c2f0330f50e1f3c8600dd46cc33), Trace(trace_id=tr-5a3eb3b093a1d1a939d2c3c67875e92e), Trace(trace_id=tr-69c9134b98bd36bebb309f3d17c17739)]

## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.

**RESPUESTA** 
evaluation_run_id: d44e316929764c10b03970e486a238f2

Número de casos: 5

Porcentaje que pasa 100%


In [0]:
#mlflow.end_run()

In [0]:
with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df))
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean()))
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

2026/10/02 21:30:34 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


{'evaluation_run_id': 'd44e316929764c10b03970e486a238f2', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

run_id,tags.agent.version,metrics.pass_rate
d44e316929764c10b03970e486a238f2,v1-deterministic,1.0
7887597d5dcd4a14a496ae426ec7baa8,v1-deterministic,null
fe3419f6346345688e694e97e5a7f720,v1-deterministic,null


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente.


In [0]:
EVALUATION_RUN_ID


'd44e316929764c10b03970e486a238f2'


![image_1790977469299.png](./image_1790977469299.png "image_1790977469299.png")






![image_1790977427004.png](./image_1790977427004.png "image_1790977427004.png")

EVALUATION_RUN_ID:
d44e316929764c10b03970e486a238f2

pass_rate:
100 %

Cubriría el riesgo de que el agente clasifique incorrectamente preguntas relacionadas con secretos y lo que haría es ampliar los casos de evaluación para que cada pregunta se dirige a la ruta y al contexto adecuados